# Bounded blend calibration and submission

Test whether the selected Random Forest and histogram-boosting components are being combined suboptimally. The feature policy, component settings, frozen five development folds and competition metric remain unchanged.

Two fixed probability weights bracket the incumbent equal vote. A regularised multinomial stack is also evaluated, but its combiner is fitted from inner out-of-fold probabilities separately inside every outer fold. The reserved local test is not loaded, inspected or scored for model selection.

In [1]:
from pathlib import Path
import hashlib
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != 'notebooks' or NOTEBOOK_DIR.parent.name != 'stage-1-pump-it-up':
    raise RuntimeError('Run this notebook from stage-1-pump-it-up/notebooks.')

STAGE_DIR = NOTEBOOK_DIR.parent
DATA_DIR = STAGE_DIR / 'data'
SRC_DIR = STAGE_DIR / 'src'
SUBMISSION_DIR = STAGE_DIR / 'submissions' / '2026-08-21-blend-calibration'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from blend_submission import fit_calibrated_stack_for_competition
from blend_submission import fit_weighted_vote_for_competition
from data_partitioning import make_cross_validation, partition_modelling_data
from final_model import write_validated_submission
from model_evaluation import evaluate_calibrated_stack
from model_evaluation import evaluate_equal_weight_soft_vote
from model_evaluation import evaluate_initial_histogram_boosting
from model_evaluation import evaluate_random_forest
from model_evaluation import evaluate_weighted_soft_vote
from modelling_data import prepare_modelling_data

## Reconstruct only the permitted data roles

The frozen development membership supplies all selection evidence. The complete labelled `original` frame and separate `competition` frame are retained only for a conditional full-data refit after selection.

In [2]:
raw_original = pd.read_csv(DATA_DIR / 'TrainingSetValues.csv')
labels_original = pd.read_csv(DATA_DIR / 'TrainingSetLabels.csv')
raw_competition = pd.read_csv(DATA_DIR / 'TestSetValues.csv')
submission_template = pd.read_csv(DATA_DIR / 'SubmissionFormat.csv')

modelling_data = prepare_modelling_data(
    raw_original,
    labels_original,
    raw_competition,
)
partitioned_data = partition_modelling_data(modelling_data)
cross_validation = make_cross_validation(partitioned_data)

print(f'Development rows: {len(partitioned_data.y_development):,}')
print(f'Frozen folds: {partitioned_data.validation_folds.nunique()}')
print(f'Local-test rows kept outside selection: {len(partitioned_data.y_local_test):,}')

Development rows: 47,520
Frozen folds: 5
Local-test rows kept outside selection: 11,880


## Recreate the incumbent component evidence

Fit both unchanged components on each frozen fold. Their aligned out-of-fold probabilities support the incumbent equal vote and the two predeclared 40:60 alternatives without additional fitting.

In [3]:
random_forest_evaluation = evaluate_random_forest(
    partitioned_data,
    cross_validation,
)
histogram_boosting_evaluation = evaluate_initial_histogram_boosting(
    partitioned_data,
    cross_validation,
)
incumbent_evaluation = evaluate_equal_weight_soft_vote(
    partitioned_data,
    cross_validation,
    random_forest_evaluation,
    histogram_boosting_evaluation,
    model_name='50% Random Forest + 50% boosting',
)
rf_40_evaluation = evaluate_weighted_soft_vote(
    partitioned_data,
    cross_validation,
    random_forest_evaluation,
    histogram_boosting_evaluation,
    weights=[0.4, 0.6],
    model_name='40% Random Forest + 60% boosting',
)
rf_60_evaluation = evaluate_weighted_soft_vote(
    partitioned_data,
    cross_validation,
    random_forest_evaluation,
    histogram_boosting_evaluation,
    weights=[0.6, 0.4],
    model_name='60% Random Forest + 40% boosting',
)

weighted_fold_accuracy = pd.DataFrame({
    evaluation.model_name: evaluation.fold_metrics['accuracy']
    for evaluation in (
        incumbent_evaluation,
        rf_40_evaluation,
        rf_60_evaluation,
    )
})
print(weighted_fold_accuracy.map(lambda value: f'{value:.2%}').to_string())
print('\nMean accuracy')
print(weighted_fold_accuracy.mean().map(lambda value: f'{value:.3%}').to_string())

Completed Random Forest fold 1/5 in 17.5 seconds.


Completed Random Forest fold 2/5 in 17.7 seconds.


Completed Random Forest fold 3/5 in 17.2 seconds.


Completed Random Forest fold 4/5 in 17.6 seconds.


Completed Random Forest fold 5/5 in 17.4 seconds.


...

Completed histogram boosting fold 2/5 in 13.9 seconds.


Completed histogram boosting fold 3/5 in 14.1 seconds.


Completed histogram boosting fold 4/5 in 14.8 seconds.


Completed histogram boosting fold 5/5 in 15.2 seconds.


                50% Random Forest + 50% boosting 40% Random Forest + 60% boosting 60% Random Forest + 40% boosting
validation_fold                                                                                                   
1                                         81.33%                           81.49%                           81.17%
2                                         81.39%                           81.24%                           81.26%
3                                         80.86%                           80.98%                           80.72%
4                                         82.06%                           82.05%                           81.87%
5                                         81.37%                           81.37%                           81.33%

Mean accuracy
50% Random Forest + 50% boosting    81.402%
40% Random Forest + 60% boosting    81.425%
60% Random Forest + 40% boosting    81.271%


## Evaluate the calibrated stack

For each frozen outer fold, the stack performs a fresh stratified four-fold split of the outer-training rows. The meta-model is fitted only from those inner out-of-fold component probabilities; both components are then refitted on all outer-training rows before predicting the outer validation fold. This is deliberately more expensive than fitting a combiner directly to the already-created outer OOF table, because that shortcut would contaminate the meta-level comparison.

In [4]:
calibrated_stack_evaluation = evaluate_calibrated_stack(
    partitioned_data,
    cross_validation,
)
print('Calibrated-stack fold metrics')
print(
    calibrated_stack_evaluation.fold_metrics
    .map(lambda value: f'{value:.2%}')
    .to_string()
)
print('\nStack diagnostics')
print(calibrated_stack_evaluation.diagnostics.round(2).to_string())

Completed calibrated stack fold 1/5 in 130.7 seconds.


Completed calibrated stack fold 2/5 in 131.2 seconds.


Completed calibrated stack fold 3/5 in 129.7 seconds.


Completed calibrated stack fold 4/5 in 129.8 seconds.


Completed calibrated stack fold 5/5 in 130.9 seconds.
Calibrated-stack fold metrics
                accuracy recall: functional recall: functional needs repair recall: non functional
validation_fold                                                                                   
1                 81.33%             89.71%                          32.90%                 78.64%
2                 81.34%             90.49%                          27.06%                 78.69%
3                 80.87%             89.21%                          30.97%                 78.53%
4                 82.15%             90.58%                          33.14%                 79.52%
5                 81.32%             90.06%                          29.52%                 78.78%

Stack diagnostics
                 inner_folds  meta_features  combiner_iterations  coefficient_l2_norm  fit_seconds  predict_seconds  total_seconds
validation_fold                                                          

## Apply the predeclared selection gate

A challenger must beat the incumbent development-fold mean and improve accuracy in at least three of the five folds. The highest-mean challenger satisfying both conditions is selected. The local test plays no part in this decision.

In [5]:
challengers = (
    rf_40_evaluation,
    rf_60_evaluation,
    calibrated_stack_evaluation,
)
incumbent_accuracy = incumbent_evaluation.fold_metrics['accuracy']
selection_rows = []
for evaluation in challengers:
    fold_change = evaluation.fold_metrics['accuracy'] - incumbent_accuracy
    mean_accuracy = evaluation.fold_metrics['accuracy'].mean()
    mean_change = mean_accuracy - incumbent_accuracy.mean()
    fold_wins = int(fold_change.gt(0).sum())
    selection_rows.append({
        'candidate': evaluation.model_name,
        'mean_accuracy': mean_accuracy,
        'mean_change': mean_change,
        'fold_wins': fold_wins,
        'worst_fold_change': fold_change.min(),
        'passes_gate': mean_change > 0 and fold_wins >= 3,
    })

selection = pd.DataFrame(selection_rows).set_index('candidate')
eligible = selection.query('passes_gate').sort_values(
    'mean_accuracy',
    ascending=False,
)
selected_name = None if eligible.empty else eligible.index[0]

formatted_selection = selection.copy()
for column in ('mean_accuracy', 'mean_change', 'worst_fold_change'):
    formatted_selection[column] = formatted_selection[column].map(
        lambda value: f'{value:.3%}'
    )
print(formatted_selection.to_string())
print(f'\nSelected challenger: {selected_name or "none"}')

                                 mean_accuracy mean_change  fold_wins worst_fold_change  passes_gate
candidate                                                                                           
40% Random Forest + 60% boosting       81.425%      0.023%          2           -0.147%        False
60% Random Forest + 40% boosting       81.271%     -0.130%          0           -0.189%        False
calibrated stack                       81.406%      0.004%          2           -0.042%        False

Selected challenger: none


In [6]:
comparison_evaluations = (
    incumbent_evaluation,
    rf_40_evaluation,
    rf_60_evaluation,
    calibrated_stack_evaluation,
)
mean_metrics = pd.DataFrame({
    evaluation.model_name: evaluation.metric_summary['mean']
    for evaluation in comparison_evaluations
})
print(mean_metrics.map(lambda value: f'{value:.2%}').to_string())

                                50% Random Forest + 50% boosting 40% Random Forest + 60% boosting 60% Random Forest + 40% boosting calibrated stack
metric                                                                                                                                             
accuracy                                                  81.40%                           81.42%                           81.27%           81.41%
recall: functional                                        90.09%                           90.41%                           89.62%           90.01%
recall: functional needs repair                           33.64%                           33.30%                           34.40%           30.72%
recall: non functional                                    78.15%                           77.84%                           78.34%           78.83%


## Conditional full-data refit

Generate one competition candidate only when the selection gate passes. The selected method is reconstructed unchanged on all 59,400 labelled original rows. The output is then reloaded and checked against the supplied template before its hash is recorded.

In [7]:
filename_by_candidate = {
    '40% Random Forest + 60% boosting': '01-rf-40-histogram-boosting-60.csv',
    '60% Random Forest + 40% boosting': '01-rf-60-histogram-boosting-40.csv',
    'calibrated stack': '01-calibrated-stack.csv',
}
candidate_result = None
if selected_name is None:
    print('No challenger passed the selection gate; no submission was generated.')
else:
    if selected_name == 'calibrated stack':
        competition_prediction = fit_calibrated_stack_for_competition(
            modelling_data,
            submission_template,
        )
    else:
        random_forest_weight = (
            0.4 if selected_name.startswith('40%') else 0.6
        )
        competition_prediction = fit_weighted_vote_for_competition(
            modelling_data,
            submission_template,
            random_forest_weight=random_forest_weight,
        )

    candidate_path = write_validated_submission(
        competition_prediction,
        SUBMISSION_DIR / filename_by_candidate[selected_name],
    )
    candidate_hash = hashlib.sha256(candidate_path.read_bytes()).hexdigest()
    candidate_result = pd.Series({
        'selected_method': selected_name,
        'path': str(candidate_path),
        'rows': len(competition_prediction.submission),
        'sha256': candidate_hash,
    })
    print(candidate_result.to_string())
    print('\nFull-data fit and prediction seconds')
    print(competition_prediction.component_seconds.round(1).to_string())
    print('\nCompetition prediction shares')
    print(
        competition_prediction.class_shares
        .map(lambda value: f'{value:.2%}')
        .to_string()
    )

No challenger passed the selection gate; no submission was generated.


## Interpretation

Treat the public leaderboard result, if this candidate is uploaded, as external feedback rather than a replacement for the frozen-fold evidence. The next independent submission hypothesis remains the conservative high-cardinality feature-family comparison.